# Music Genre Classifier: Feature Extraction

This notebook runs entirely on Google Colab's compute, not your local machine.

It downloads the FMA small dataset, extracts audio features with librosa, and checkpoints progress to your Google Drive after every batch. If the Colab runtime disconnects or times out, just rerun this notebook and it will resume from where it left off instead of starting over.

Steps:
1. Mount Google Drive
2. Download FMA small (about 7.2GB)
3. Extract features in batches, saving checkpoints to Drive
4. Combine into a final features CSV you download for local use

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/music_genre_classifier'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint directory ready at', CHECKPOINT_DIR)

In [ ]:
!pip install -q librosa pandas numpy tqdm

# Clone the repo so this notebook reuses the exact same extract_features()
# function as the local Streamlit app, instead of keeping a second hand-copied
# version in sync.
import os
REPO_DIR = '/content/music-genre-classifier'
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/bcherukuri2004/music-genre-classifier.git {REPO_DIR}

import sys
sys.path.append(REPO_DIR)

In [ ]:
import os

DATA_DIR = '/content/fma_small'
METADATA_DIR = '/content/fma_metadata'

if not os.path.exists(DATA_DIR):
    !wget -q https://os.unil.cloud.switch.ch/fma/fma_small.zip -O /content/fma_small.zip
    !unzip -q /content/fma_small.zip -d /content
    print('Audio files downloaded and extracted')
else:
    print('Audio files already present, skipping download')

if not os.path.exists(METADATA_DIR):
    !wget -q https://os.unil.cloud.switch.ch/fma/fma_metadata.zip -O /content/fma_metadata.zip
    !unzip -q /content/fma_metadata.zip -d /content
    print('Metadata downloaded and extracted')
else:
    print('Metadata already present, skipping download')

In [ ]:
import pandas as pd

tracks = pd.read_csv(f'{METADATA_DIR}/tracks.csv', index_col=0, header=[0, 1])

small = tracks[tracks[('set', 'subset')] == 'small']
genres = small[('track', 'genre_top')]

track_genre_map = genres.dropna().to_dict()
print(f'Found {len(track_genre_map)} labeled tracks in the small subset')
print('Genres:', sorted(set(track_genre_map.values())))

In [ ]:
import os
from scripts.features import extract_features

def get_audio_path(track_id, base_dir=DATA_DIR):
    tid_str = f'{track_id:06d}'
    return os.path.join(base_dir, tid_str[:3], f'{tid_str}.mp3')

In [ ]:
import librosa
import json
from tqdm import tqdm

CHECKPOINT_FILE = f'{CHECKPOINT_DIR}/features_checkpoint.csv'
FAILED_LOG = f'{CHECKPOINT_DIR}/failed_tracks.json'

if os.path.exists(CHECKPOINT_FILE):
    done_df = pd.read_csv(CHECKPOINT_FILE)
    done_ids = set(done_df['track_id'].tolist())
    print(f'Resuming, {len(done_ids)} tracks already processed')
else:
    done_df = pd.DataFrame()
    done_ids = set()
    print('Starting fresh extraction run')

failed_tracks = []
if os.path.exists(FAILED_LOG):
    with open(FAILED_LOG, 'r') as f:
        failed_tracks = json.load(f)

remaining = {tid: g for tid, g in track_genre_map.items() if tid not in done_ids}
print(f'{len(remaining)} tracks remaining')

In [ ]:
BATCH_SIZE = 50
batch_results = []

remaining_items = list(remaining.items())

for idx, (track_id, genre) in enumerate(tqdm(remaining_items)):
    path = get_audio_path(track_id)

    if not os.path.exists(path):
        failed_tracks.append({'track_id': track_id, 'reason': 'file_not_found'})
        continue

    try:
        feats = extract_features(path)
        feats['track_id'] = track_id
        feats['genre'] = genre
        batch_results.append(feats)
    except Exception as e:
        failed_tracks.append({'track_id': track_id, 'reason': str(e)})

    if len(batch_results) >= BATCH_SIZE:
        batch_df = pd.DataFrame(batch_results)
        header = not os.path.exists(CHECKPOINT_FILE)
        batch_df.to_csv(CHECKPOINT_FILE, mode='a', header=header, index=False)
        with open(FAILED_LOG, 'w') as f:
            json.dump(failed_tracks, f)
        batch_results = []

if batch_results:
    batch_df = pd.DataFrame(batch_results)
    header = not os.path.exists(CHECKPOINT_FILE)
    batch_df.to_csv(CHECKPOINT_FILE, mode='a', header=header, index=False)
    with open(FAILED_LOG, 'w') as f:
        json.dump(failed_tracks, f)

print('Extraction complete for this run')
print(f'Failed tracks so far: {len(failed_tracks)}')

## If your session disconnects

Just rerun every cell from the top. The checkpoint cell will detect the tracks already saved in `features_checkpoint.csv` on your Drive and skip them, so you only process what is left. Nothing is lost.

In [ ]:
final_df = pd.read_csv(CHECKPOINT_FILE)
print(f'Total tracks with extracted features: {len(final_df)}')
print(final_df['genre'].value_counts())
final_df.to_csv(f'{CHECKPOINT_DIR}/features_final.csv', index=False)
print(f'Saved final feature set to {CHECKPOINT_DIR}/features_final.csv')
print('Download this file from Drive and place it in your local repo data folder')